# Cycle 3 - Comparaison de modèles

Cinq modèles de familles différentes sont comparés avec leurs **paramètres par défaut**, dans les mêmes conditions : même prétraitement, même cible en log et même validation croisée temporelle (5 plis) qu'au cycle 2.

Dans MLflow, chaque modèle est un run enfant regroupé sous le run « Cycle 3 - Comparaison des modèles ». La vue de comparaison de MLflow permet d'afficher les scores des modèles côte à côte sous forme de graphique.

In [1]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

import notebooks.utils as utils
from agritech_feature_engineering import AgritechFeatureEngineering

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Agriculture_optimization")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1795094493928, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1795094493928, lifecycle_stage='active', name='Agriculture_optimization', tags={}, trace_location=None, workspace='default'>

## Données

Même découpage temporel que dans les cycles précédents.

In [3]:
dataset = pd.read_csv('../data/rafined/dataset.csv')
dataset = dataset.drop(columns=['Area ISO Code', 'Area ISO Alpha-2 Code', 'Area ISO Alpha-3 Code'])

X_train, X_test, y_train, y_test = utils.test_train_time_split(dataset)
X_train = X_train.sort_values('Year')  # indispensable pour TimeSeriesSplit
y_train = y_train.loc[X_train.index]

Année de césure calculée : 2009
Proportion réelle train : 78.73% (20349 lignes)
Proportion réelle test  : 21.27% (5499 lignes)


## Prétraitement

Identique aux cycles 1 et 2.

In [4]:
numeric_features = [
    'avg_temp', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'pesticides_kg_ha',
    'aridity', 'avg_temp_square',
    'missing_pesticide_value', 'missing_rain_value', 'missing_temp_value',
]

preprocessor = ColumnTransformer([
    ('area_encoding', TargetEncoder(target_type='continuous', random_state=42), ['Area']),
    ('item_encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Item']),
    ('num_pipeline', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_features),
])

## Modèles comparés

- **Ridge** : régression linéaire régularisée, référence linéaire.
- **K plus proches voisins** : prédit à partir des situations les plus similaires.
- **Random Forest** : moyenne de nombreux arbres de décision indépendants.
- **Gradient boosting** (classique et par histogrammes) : arbres construits les uns après les autres, chacun corrigeant les erreurs des précédents.

In [5]:
models = {
    'ridge': Ridge(random_state=42),
    'k_neighbors': KNeighborsRegressor(),
    'random_forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'hist_gradient_boosting': HistGradientBoostingRegressor(random_state=42),
    'gradient_boosting': GradientBoostingRegressor(random_state=42),
}

## Comparaison en validation croisée temporelle

On garde aussi le R² sur le train : un grand écart avec le R² de validation signale du surapprentissage.

In [6]:
scoring = {'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'}

resultats = []
with mlflow.start_run(run_name="Cycle 3 - Comparaison des modèles"):
    mlflow.set_tag('cycle', '3_comparaison')

    for nom, model in models.items():
        pipeline = Pipeline([
            ('features', AgritechFeatureEngineering()),
            ('preprocessor', preprocessor),
            ('model', model),
        ])
        model_log = TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)

        cv_results = cross_validate(model_log, X_train, y_train, cv=TimeSeriesSplit(5), scoring=scoring, return_train_score=True)
        metrics = {
            'r2': cv_results['test_r2'].mean(),
            'mae': -cv_results['test_mae'].mean(),
            'rmse': -cv_results['test_rmse'].mean(),
            'train_r2': cv_results['train_r2'].mean(),
            'fit_time': cv_results['fit_time'].mean(),
        }

        # un run enfant par modèle, rattaché au run parent
        with mlflow.start_run(run_name=f"Cycle 3 - {nom}", nested=True):
            mlflow.set_tag('cycle', '3_comparaison')
            mlflow.log_param('model', nom)
            mlflow.log_metrics(metrics)

        resultats.append({'modele': nom, **metrics})

resultats = pd.DataFrame(resultats).sort_values('rmse')
resultats.round(3)

/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/1

🏃 View run Cycle 3 - ridge at: http://127.0.0.1:5000/#/experiments/1/runs/a23d341fb3454778a41f52cfb8039a9e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/1

🏃 View run Cycle 3 - k_neighbors at: http://127.0.0.1:5000/#/experiments/1/runs/7458de53ab5b47b8a49cae6a09cb95a6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/1

🏃 View run Cycle 3 - random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/42ba8611d22b4f0d8412a5a2067ba12f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/1

🏃 View run Cycle 3 - hist_gradient_boosting at: http://127.0.0.1:5000/#/experiments/1/runs/45c9ab9a576c4b59983730129a412653
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/1

🏃 View run Cycle 3 - gradient_boosting at: http://127.0.0.1:5000/#/experiments/1/runs/e3203649decd414ebdb90e67cb3aa5f3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Cycle 3 - Comparaison des modèles at: http://127.0.0.1:5000/#/experiments/1/runs/f24c0b24569a46a1bfc988493915eedd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,modele,r2,mae,rmse,train_r2,fit_time
2,random_forest,0.874,13376.859,25832.433,0.966,0.485
1,k_neighbors,0.847,14598.023,28402.079,0.923,0.020
3,hist_gradient_boosting,0.798,18690.195,32836.369,0.865,1.003
4,gradient_boosting,0.722,22857.201,38544.601,0.771,1.011
0,ridge,0.653,25360.063,43081.198,0.684,0.024


## Conclusion du cycle 3

Pour rappel, la baseline naïve obtient une MAE de 16 461 hg/ha et une RMSE de 32 646 hg/ha.

- **Les modèles non linéaires font beaucoup mieux** : la Random Forest divise presque par deux l'erreur de Ridge (MAE de 13 377 contre 25 360 hg/ha). Ridge obtient les mêmes scores que la régression linéaire du cycle 2 : le problème vient bien de la forme du modèle, qui ne peut pas apprendre des effets différents selon la culture et le pays.
- **La Random Forest est la meilleure sur les trois métriques** (R² 0.874, MAE 13 377, RMSE 25 832 hg/ha) et la seule à battre nettement la baseline (environ −19 % de MAE et −21 % de RMSE). Les K plus proches voisins font à peine mieux que la baseline, et les deux gradient boostings font moins bien avec leurs réglages par défaut.
- **La Random Forest surapprend un peu** : R² de 0.966 sur le train contre 0.874 en validation. Limiter la profondeur des arbres ou imposer plusieurs lignes par feuille sont des pistes pour l'optimisation.
- Les deux évaluations ne portent pas sur les mêmes années : le gain face à la baseline sera vérifié sur le jeu de test au cycle 5.

**Choix pour le cycle 4** : la Random Forest est retenue pour l'optimisation de ses hyperparamètres. Elle permet aussi de mesurer l'importance de chaque variable, utile pour expliquer les prédictions.